# KAIROS — proof of concept

Four steps, in the order the pipeline runs them.

1. **Extract** a valve passport from the real de-identified notes, and report aggregates only.
2. **Stage** an echo against the patient's own reference study under VARC-3.
3. **Build** a landmark dataset from a synthetic scenario and fit the cause-specific model.
4. **Evaluate** against the comparator ladder.

> Steps 3 and 4 run on **explicitly synthetic** scenarios. Every number they produce is evidence
> about the software, not about patients. No clinical accuracy is demonstrated or claimed.

Requires the package installed (`pip install -e ".[dev]"`). The three supplied spreadsheets must
sit at the repository root; they are gitignored and never committed.


## 0. Setup


In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
pd.set_option('display.width', 120)
print('repository root:', ROOT)


## 1. Extraction on the real notes

`prepare_notes` applies the deleted-status exclusion. 215 rows become 202 across 117 patients.


In [ ]:
from kairos.passport import load_notes, build_passport

notes = load_notes()
print(f'{len(notes)} notes, {notes["Profile Key"].nunique()} patients')
passport = build_passport()
passport = passport[0] if isinstance(passport, tuple) else passport
print(f'{len(passport)} patient rows in the passport')


### Aggregates only

Patient-level rows stay out of this notebook's output. Counts are printed with small cells
suppressed, exactly as they are written to `data/derived/aggregates/`.


In [ ]:
print(passport['route'].value_counts(dropna=False).to_string())
print()
for col, label in [('canonical_model','valve model named'), ('size_mm','label size'), 
                   ('implant_year','implant year stated')]:
    if col in passport.columns:
        print(f'{label:24s} {passport[col].notna().sum():3d} of {len(passport)}')


### The extraction trap that changed a published number

Transcatheter operative reports carry a structured field reading `Valve in Valve: No`.
Matching the phrase counts those patients as having had the event the field denies.
The guard below rejects any match that falls inside such a field.


In [ ]:
from kairos.passport import EVENT_PATS, negated_field_spans, in_negated_field

denied = 'PROCEDURE: TAVR\nValve in Valve: No\nAccess: transfemoral'
prose  = 'Underwent transfemoral TAVR (valve-in-valve) with a 26 mm Evolut FX.'

def fires(key, text):
    spans = negated_field_spans(text)
    return any(not in_negated_field(m.start(), spans) for m in EVENT_PATS[key].finditer(text))

print('phrase present in the denied form field :', bool(EVENT_PATS['ViV'].search(denied)))
print('counted as an event                     :', fires('ViV', denied))
print('genuine prose still counted             :', fires('ViV', prose))


In [ ]:
# Effect on the corpus: the reported count falls from 10 patients to 4.
before = after = 0
for _, r in notes.iterrows():
    t = str(r['Notes'])
    if EVENT_PATS['ViV'].search(t): before += 1
    if fires('ViV', t): after += 1
print(f'notes matching the phrase: {before}   notes surviving the guard: {after}')


## 2. VARC-3 staging against the patient's own reference study

Change-based staging needs the patient's baseline. Published normal values give context and
can never replace it. An input that cannot be resolved returns `uncertain`, never a negative.


In [ ]:
from kairos.varc3 import stage_hvd
import inspect; print(inspect.signature(stage_hvd))


In [ ]:
# A trajectory that rises but crosses no threshold: the case the model exists to find.
reference = dict(mean_gradient=11.0, dvi=0.48, eoa=1.7, regurgitation='none')
year_four = dict(mean_gradient=19.0, dvi=0.37, eoa=1.25, regurgitation='trace')
print('reference:', reference)
print('year four:', year_four)
print()
print('gradient rose', year_four['mean_gradient'] - reference['mean_gradient'],
      'mmHg. Stage 2 needs a rise of 10 to reach 20.')


## 3. Landmark dataset and the cause-specific model

One row per patient per prediction time, every feature as known at that time. The builder
enforces the leakage rules: nothing dated after the landmark, patients who met the endpoint
leave the risk set, and all rows of one patient stay together in resampling.


In [ ]:
from kairos.simulation.scenarios import load_scenarios

scenarios = load_scenarios()
print('available scenarios:')
for name in scenarios: print('  -', name)


In [ ]:
# Generate one scenario and build its landmark dataset.
from kairos.simulation.generators import generate
from kairos.modelling.landmark import build_landmarks

cohort = generate('gradual_stenotic', seed=20260917)
landmarks = build_landmarks(cohort)
print('landmark rows:', len(landmarks))
print('distinct patients:', landmarks['patient_id'].nunique())
print('rows per patient:', round(len(landmarks) / landmarks['patient_id'].nunique(), 1))


In [ ]:
from kairos.modelling.cause_specific import CauseSpecificCoxModel
from kairos.modelling.cif import combine_cause_specific

model = CauseSpecificCoxModel()
model.fit(landmarks)
print('causes fitted:', list(model.fitters)) if hasattr(model, 'fitters') else print(model)


### The three probabilities

One minus the deterioration risk is **not** the chance of being alive with a working valve.
It includes everyone who died first. The model reports the three states separately.


In [ ]:
pred = model.predict(landmarks.head(1))
print(pred)


## 4. Evaluation against the comparator ladder

The reference is valve age and type alone. The comparison that decides clinical usefulness is
against the guideline threshold rule, which is implemented in `kairos.comparators.varc3_hvd`.


In [ ]:
print((ROOT / 'artifacts' / 'ladder_summary.md').read_text()[:1800])


### Reproducing everything

```bash
make test      # unit, contract and service tests
make quick     # scenarios, training and evaluation into artifacts/
```

Figures land in `artifacts/figures/`, the ladder table in `artifacts/ladder_summary.md`.
`scripts/privacy_scan.py` runs in CI and blocks any patient-level row from reaching
version control.
